# GMM clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> gmm_clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV

from utils.clustering_utils import (
    cluster_summaries,
    predict_clustering_labels,
    predict_clustering_proba,
    print_cluster_stability,
    print_cluster_statistics,
    print_clustering_info,
)
from utils.misc_utils import turn_cats_to_continuous
from utils.plotting_utils import make_plot_cv, make_plot_mixture_weights
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

# Set outputs folder
outputs_folder = "outputs/gmm_clustering"

In [ ]:
# Set up models
def gmm_neg_bic(estimator: GaussianMixture, data: np.typing.NDArray | pd.DataFrame):
    return -estimator.bic(data)


param_grid = [
    {"n_components": range(1, 50), "covariance_type": ["spherical"]},
    {"n_components": range(1, 50), "covariance_type": ["tied"]},
    {"n_components": range(1, 50), "covariance_type": ["diag"]},
    {"n_components": range(1, 50), "covariance_type": ["full"]},
]
gmm_discov = GaussianMixture(random_state=np.random.RandomState(np.random.default_rng(2352).bit_generator))
grid_search_discov = GridSearchCV(gmm_discov, param_grid=param_grid, scoring=gmm_neg_bic, n_jobs=-1, refit=True)

gmm_valid = GaussianMixture(random_state=np.random.RandomState(np.random.default_rng(54645).bit_generator))

# Fit models
grid_search_discov.fit(data_discov_mfa)
print(f"Best parameters after cross-validation: {grid_search_discov.best_params_}")
plot_cv = make_plot_cv(grid_search_discov, group="covariance_type")
plot_cv.show()
save_plot(plot_cv, Path(outputs_folder, "plot_cv.png"), dpi=150)

gmm_valid.set_params(**grid_search_discov.best_params_)
gmm_valid.fit(data_valid_mfa)

model_discov = grid_search_discov.best_estimator_
model_valid = gmm_valid

In [ ]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_mfa)
labels_valid = predict_clustering_labels(model_valid, data_valid_mfa)

# Apply models to predict probabilities
proba_discov = predict_clustering_proba(model_discov, data_discov_mfa)
proba_valid = predict_clustering_proba(model_valid, data_valid_mfa)

# Save predictions
proba_discov.assign(**{"label": labels_discov}).to_parquet(
    Path(outputs_folder, "cluster_pred_discov.parquet"), index=False
)

proba_valid.assign(**{"label": labels_valid}).to_parquet(
    Path(outputs_folder, "cluster_pred_valid.parquet"), index=False
)

### Display model information

In [ ]:
# Print information about clusterings
print("Discovery:    ", end="")
print_clustering_info(labels_discov)
print("Validation:   ", end="")
print_clustering_info(labels_valid)

# Plot mixture weights
plot_discov_weights = make_plot_mixture_weights(model_discov)
plot_valid_weights = make_plot_mixture_weights(model_valid)

# save_plot(plot_discov_weights, Path(outputs_folder, "plot_discov_weights.png"), dpi=150)
# save_plot(plot_valid_weights, Path(outputs_folder, "plot_valid_weights.png"), dpi=150)

# Combine the two plots
plot_discov_weights += p9.labs(title="Discovery")
plot_valid_weights += p9.labs(title="Validation")
plot_all_weights = (plot_discov_weights / plot_valid_weights) & p9.theme(figure_size=(5, 5))

plot_all_weights.show()

save_plot(plot_all_weights, Path(outputs_folder, "plot_all_weights.png"), dpi=150)

### Get cluster summaries

In [ ]:
treat_cat_as_continuous = data_discov_full.meta.loc[lambda x: x["type"].isin(["binary", "ordinal"]), "name"]
data_discov_cont = turn_cats_to_continuous(data_discov_full.data, treat_cat_as_continuous).select_dtypes(
    exclude="category"
)
data_valid_cont = turn_cats_to_continuous(data_valid_full.data, treat_cat_as_continuous).select_dtypes(
    exclude="category"
)

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_discov_summaries = cluster_summaries(data_discov_cont, labels_discov, agg_func=["mean", "std"])  # "min", "max"
cluster_discov_summaries

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_valid_summaries = cluster_summaries(data_valid_cont, labels_valid, agg_func=["mean", "std"])  # "min", "max"
cluster_valid_summaries

### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Compare clusterings
print_cluster_statistics(
    cluster_labels_list=[labels_discov, labels_valid],
    datas=[data_discov_mfa, data_valid_mfa],
    labels=["discovery", "validation"],
)